In [ ]:
import pandas as pd
import numpy as np

df = pd.read_csv("lego.csv")
df.info()



<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3648 entries, 0 to 3647
Data columns (total 13 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   set_id       3648 non-null   object
 1   name         3648 non-null   object
 2   year         3648 non-null   int64 
 3   theme        3648 non-null   object
 4   subtheme     3648 non-null   object
 5   themeGroup   3648 non-null   object
 6   category     3648 non-null   object
 7   pieces       3648 non-null   int64 
 8   age          3648 non-null   int64 
 9   price        3648 non-null   object
 10  imageURL     3648 non-null   object
 11  agerange     3648 non-null   object
 12  price range  3648 non-null   object
dtypes: int64(3), object(10)
memory usage: 370.6+ KB


In [ ]:
# Define categories
regions = ["USA", "Europe", "Asia-Pacific", "Australia"]
channels = ["Online", "Retail"]

# Create all combinations
expanded_df = df.assign(key=1).merge(
    pd.DataFrame({
        "Region": np.repeat(regions, len(channels)),
        "Sales_Channel": channels * len(regions),
        "key": 1
    }),
    on="key"
).drop("key", axis=1)
expanded_df.head()




,set_id,name,year,theme,subtheme,themeGroup,category,pieces,age,price,imageURL,agerange,price range,Region,Sales_Channel
0,8683-4,Circus Clown,2010,Collectable Minifigures,Series 1,Miscellaneous,Normal,6,5,$1.99,https://images.brickset.com/sets/images/8683-4...,5 to 9,$,USA,Online
1,8683-4,Circus Clown,2010,Collectable Minifigures,Series 1,Miscellaneous,Normal,6,5,$1.99,https://images.brickset.com/sets/images/8683-4...,5 to 9,$,USA,Retail
2,8683-4,Circus Clown,2010,Collectable Minifigures,Series 1,Miscellaneous,Normal,6,5,$1.99,https://images.brickset.com/sets/images/8683-4...,5 to 9,$,Europe,Online
3,8683-4,Circus Clown,2010,Collectable Minifigures,Series 1,Miscellaneous,Normal,6,5,$1.99,https://images.brickset.com/sets/images/8683-4...,5 to 9,$,Europe,Retail
4,8683-4,Circus Clown,2010,Collectable Minifigures,Series 1,Miscellaneous,Normal,6,5,$1.99,https://images.brickset.com/sets/images/8683-4...,5 to 9,$,Asia-Pacific,Online


In [ ]:
# Estimated Units
pieces = expanded_df["pieces"]

expanded_df["Units_Sold"] = np.select(
    [
        pieces < 100,
        (pieces >= 100) & (pieces < 500),
        (pieces >= 500) & (pieces < 1000),
        pieces >= 1000
    ],
    [
        np.random.randint(10000, 50000, len(expanded_df)),
        np.random.randint(5000, 20000, len(expanded_df)),
        np.random.randint(2000, 10000, len(expanded_df)),
        np.random.randint(1000, 5000, len(expanded_df))
    ]
)

expanded_df.head()

,set_id,name,year,theme,subtheme,themeGroup,category,pieces,age,price,imageURL,agerange,price range,Region,Sales_Channel,Units_Sold
0,8683-4,Circus Clown,2010,Collectable Minifigures,Series 1,Miscellaneous,Normal,6,5,$1.99,https://images.brickset.com/sets/images/8683-4...,5 to 9,$,USA,Online,10313
1,8683-4,Circus Clown,2010,Collectable Minifigures,Series 1,Miscellaneous,Normal,6,5,$1.99,https://images.brickset.com/sets/images/8683-4...,5 to 9,$,USA,Retail,10516
2,8683-4,Circus Clown,2010,Collectable Minifigures,Series 1,Miscellaneous,Normal,6,5,$1.99,https://images.brickset.com/sets/images/8683-4...,5 to 9,$,Europe,Online,32763
3,8683-4,Circus Clown,2010,Collectable Minifigures,Series 1,Miscellaneous,Normal,6,5,$1.99,https://images.brickset.com/sets/images/8683-4...,5 to 9,$,Europe,Retail,45017
4,8683-4,Circus Clown,2010,Collectable Minifigures,Series 1,Miscellaneous,Normal,6,5,$1.99,https://images.brickset.com/sets/images/8683-4...,5 to 9,$,Asia-Pacific,Online,35994


In [ ]:
# Discount (vectorized)
expanded_df["Discount_%"] = np.select(
    [
        expanded_df["Region"] == "USA",
        expanded_df["Region"] == "Europe",
        expanded_df["Region"] == "Asia-Pacific",
        expanded_df["Region"] == "Australia"
    ],
    [
        np.random.choice([5, 10, 15], len(expanded_df)),
        np.random.choice([10, 15, 20], len(expanded_df)),
        np.random.choice([5, 10], len(expanded_df)),
        np.random.choice([5, 10, 15], len(expanded_df))
    ]
)

sales_df = expanded_df[[
    "set_id", "Region", "Sales_Channel",
    "Units_Sold", "Discount_%"
]]
sales_df.to_csv("sales_dataset_generated.csv", index=False)


